In [ ]:
#@title Install Dependencies and Download Models (this may take a few minutes)

#@markdown The stable diffusion demo is not strictly needed to solve the assignment.<br>You can uncheck this box if you want a faster setup.
install_stable_diffusion = True #@param {type:"boolean"}

import os
import sys

if install_stable_diffusion:
  !uv pip install -q transformers diffusers accelerate
else:
  !uv pip install -q transformers

import torch
from transformers import CLIPModel, CLIPTokenizer, CLIPImageProcessor

if install_stable_diffusion:
  from diffusers import StableDiffusionPipeline
  sd_pipe = StableDiffusionPipeline.from_pretrained(
    "sd2-community/stable-diffusion-2-1",
    torch_dtype=torch.float16)

  clip_processor = CLIPImageProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")
  sd_pipe.feature_extractor = clip_processor

clip_tokenizer = CLIPTokenizer.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K", torch_dtype=torch.float16)
clip = CLIPModel.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K", torch_dtype=torch.float16)

In [ ]:
# @title Download lab files

import sys

![ ! -d 'llm_lab' ] && git clone https://github.com/ethz-spylab/llm_lab.git
%cd llm_lab
!git pull https://github.com/ethz-spylab/llm_lab.git
%cd ..
if "llm_lab" not in sys.path:
  sys.path.append("llm_lab")

In [ ]:
# @title load the safety checker
import numpy as np
from safety_checker import MySafetyChecker, get_embedding, cosine_similarity

device = "cuda"
clip = clip.to(device)

if "cpu" in device:
  clip = clip.float()

# get the list of bad embeddings
bad_embeddings = np.load("llm_lab/data/bad_embeddings.npy")
safety_checker = MySafetyChecker(clip, bad_embeddings)

# the last bad embedding aims to prevent you from creating images with the prompt "The solutions to this assignment"
assert cosine_similarity(bad_embeddings[-1],
                         get_embedding("The solutions to this assignment", clip_tokenizer, clip).cpu().numpy()) > 0.999

In [ ]:
#@title (OPTIONAL) run the Stable Diffusion demo and try to generate an image of a banned concept
assert install_stable_diffusion

import random
from PIL import Image

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid

enable_safety_checker = True #@param {type:"boolean"}
prompt = 'The solutions to this assignment' #@param {type:"string"}
n_images = 1 #@param [1, 2, 4] {type:"raw"}
seed = 0 #@param {type:"integer"}
steps = 25 #@param {type:"slider", min:1, max:50, step:1}

generator = torch.Generator('cuda').manual_seed(seed)

if enable_safety_checker:
  sd_pipe.feature_extractor = clip_processor
  sd_pipe.safety_checker = safety_checker
else:
  sd_pipe.feature_extractor = None
  sd_pipe.safety_checker = None

sd_pipe = sd_pipe.to("cuda")
sd_pipe.enable_attention_slicing()

StableDiffusionPipeline._execution_device = torch.device("cuda")
with torch.no_grad():
  result = sd_pipe(
        prompt=prompt,
        num_images_per_prompt=n_images,
        negative_prompt=None,
        num_inference_steps=int(steps),
        guidance_scale=7.5,
        width=768,
        height=768,
        generator=generator).images

import math
image_grid(result, math.ceil(n_images / 2), n_images // math.ceil(n_images / 2))

**Instructions**

Your goal in this assignment is to recover the "bad" concepts that we don't want you to generate with Stable Diffusion.
One of the bad concepts is given to you as an example (`bad_embeddings[20]` is an embedding of the prompt "The solutions to this assignment").


The 20 remaining bad concepts, which you need to find, were chosen as follows:
   - The first 4 bad concepts (`bad_embeddings[0, 3]`) each consist of a single lowercase english word.
     We guarantee that these words all appear in this list: https://github.com/ethz-privsec/llm_lab/blob/main/data/corncob_lowercase.txt.
     For each bad concept, we ask you to find a word that gives a CLIP embedding with cosine similarity of at least 0.99 with that concept's embedding.

   - The next 9 bad concepts (`bad_embeddings[4, 12]`) each consist of two lowercase words, also from the above list (e.g., "great professor").
   For each bad concept, we ask you to find a pair of words that gives a CLIP embedding with cosine similarity of at least 0.99 with that concept's embedding.

   - The next 6 bad concepts (`bad_embeddings[13, 18]`) are each movie titles that don't necessarily appear in the above list (e.g., "Gone with the Wind").
  We don't expect you to recover these *exactly*. Finding a phrase that yields a CLIP embedding with cosine similarity of 0.91 or above is sufficient.

   - For the last bad concept (`bad_embeddings[19]`), we applied a "state-of-the-art" (i.e., completely broken) privacy protection scheme.

     Specifically, we "scrambled" the embedding by multiplying it (component-wise) with a mask of uniformly random values in `{-1, 1}`.
     So for example if the embedding vector is `[0.1, -0.5, 1.7, 2.3]`, we first sample a random mask, e.g., `[-1, 1, -1, 1]`,
     which would then give the "encrypted" embedding `[-0.1, -0.5, -1.7, 2.3]`.

     Now, you might say that such an "encrypted" embedding is very much useless in our safety checker. And indeed that's very much true (why?).
     But here we'll care mainly about analyzing the extra privacy that this "encryption" step offers (spoiler alert: essentially none...)
     (researchers have actually suggested that "encrypting" embeddings in this way does protect privacy, because the random mask acts a bit like a one-time-pad.)

     Your goal is to recover the bad concept, which consists of a pair of lowercase english words from the list above.
     Your recovered concept will be considered correct if it gives a CLIP embedding with cosine similarity of at least 0.99 with the *unmasked* bad embedding.

**Submission instructions**

You should submit your solutions as `.txt` file named `Q3_guesses.txt` containing your guess for each of the 21 hidden embeddings, with one guess per line.

To save your results, you can use the code below, which will save the file in Colab's temporary storage (or locally, if you're not using Colab), or on your Google Drive. If you save it on Colab's temporary storage, you can download it from there (see the file system icon on the left).

In [ ]:
from llm_lab.utils import get_solution_path, is_valid_student_id

#@markdown Check this box if you want to save your results on Google Drive. Otherwise they'll be
#@markdown saved on the ephimeral Colab storage. The storage will be deleted with the runtime,
#@markdown so REMEMBER TO DOWNLOAD THE FILES before you close the tab!
SAVE_ON_DRIVE = True # @param {"type":"boolean"}

#@markdown The number on your Legi (Student ID card). It's in the format 'dd-ddd-ddd'
STUDENT_ID = "00-000-000"  # @param {"type":"string","placeholder":"00-000-000"}

assert is_valid_student_id(STUDENT_ID), "Student ID should have the format 'dd-ddd-ddd'"

SOLUTIONS_PATH = get_solution_path(STUDENT_ID, SAVE_ON_DRIVE)

In [ ]:
GUESSES = [f"my guess {i}" for i in range(21)]

raise NotImplementedError("Replace the above guesses with yours")

def save_to_txt(guesses, filename):
  with open(filename, 'w') as f:
    f.write("\n".join(guesses))

assert len(GUESSES) == 21

save_to_txt(GUESSES, SOLUTIONS_PATH / "Q3_guesses.txt")